# SC-19-Ripple-XRP - Protocole Ripple et XRP Ledger

[<< Vyper](SC-18-Vyper.ipynb) | [Bitcoin Scripting >>](SC-20-Bitcoin-Scripting.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'**architecture du XRP Ledger** et son consensus UNL
2. Manipuler des **comptes et transactions XRP** via `xrpl-py`
3. Explorer les **trust lines**, **offers** et le DEX integre
4. Decouvrir les **payment channels** pour les micropaiements
5. Comprendre les **Hooks** comme equivalent smart contract du XRPL

### Prerequis

- Python 3.10+ avec `xrpl-py`
- Connexion internet (acces au Testnet XRP)

### Duree estimee : 50 minutes

---

## 1. Introduction au protocole Ripple et au XRP Ledger

### Historique

Le XRP Ledger (XRPL) a ete concu en **2012** par Jed McCaleb, Arthur Britto et David Schwartz,
avec pour objectif principal de faciliter les **paiements transfrontaliers** rapides et peu couteux.
Contrairement a Bitcoin (2009) et Ethereum (2015), le XRPL n'utilise pas de Proof-of-Work
ni de Proof-of-Stake : il repose sur un mecanisme de consensus unique base sur une
**Unique Node List (UNL)**.

### Consensus UNL vs Proof-of-Work

| Propriete | Bitcoin (PoW) | Ethereum (PoS) | XRP Ledger (UNL) |
|-----------|--------------|----------------|------------------|
| Temps de confirmation | ~10 min | ~12 sec | **3-5 sec** |
| Cout de transaction | Variable, souvent eleve | Variable (gas) | **~0.00001 XRP** |
| Consommation energetique | Tres elevee | Moderee | **Negligeable** |
| Finalite | Probabiliste | Probabiliste | **Deterministe** |
| Decentralisation | Milliers de mineurs | Milliers de validateurs | ~150 validateurs UNL |

Le consensus UNL fonctionne par **accord iteratif** : chaque validateur maintient une liste
de validateurs de confiance (la UNL). Un ledger est valide quand **80% ou plus** des validateurs
de la UNL s'accordent sur le meme ensemble de transactions.

### Concepts cles du XRPL

| Concept | Description |
|---------|-------------|
| **Account** | Adresse avec reserve de base (actuellement 10 XRP) |
| **Trust Line** | Lien bilateral pour detenir des IOU (devises personnalisees) |
| **Offer** | Ordre d'achat/vente sur le DEX integre au protocole |
| **Payment Channel** | Canal off-chain pour micropaiements en XRP |
| **Escrow** | Verrouillage conditionnel de XRP (time-lock, crypto-condition) |
| **Hook** | Programme WASM execute lors d'une transaction (smart contract) |

In [ ]:
# Comparaison des architectures blockchain
architectures = {
    "Bitcoin (2009)": {
        "Consensus": "Proof-of-Work (SHA-256)",
        "Langage SC": "Bitcoin Script (limite)",
        "TPS": "~7",
        "Finalite": "~60 min (6 confirmations)",
    },
    "Ethereum (2015)": {
        "Consensus": "Proof-of-Stake (Casper)",
        "Langage SC": "Solidity / Vyper (Turing-complet)",
        "TPS": "~15-30",
        "Finalite": "~15 min (64 slots)",
    },
    "XRP Ledger (2012)": {
        "Consensus": "UNL (Unique Node List)",
        "Langage SC": "Hooks (WASM) + primitives natives",
        "TPS": "~1500",
        "Finalite": "3-5 sec (deterministe)",
    },
}

print("COMPARAISON DES ARCHITECTURES BLOCKCHAIN")
print("=" * 65)
for chain, props in architectures.items():
    print(f"\n{chain}")
    print("-" * 40)
    for key, val in props.items():
        print(f"  {key:20s} : {val}")

### Observation

Le XRPL se distingue par son approche **pragmatique** : plutot que de chercher
la decentralisation maximale, il optimise pour la **vitesse** et le **cout**
des paiements. Les fonctionnalites de smart contracts (trust lines, offers, escrows)
sont **natives au protocole** plutot qu'executees dans une machine virtuelle generique.

Cette approche a des avantages (performance, fiabilite) et des inconvenients
(flexibilite limitee comparee a Ethereum).

---

## 2. Installation et connexion au Testnet

La bibliotheque `xrpl-py` est le SDK Python officiel pour interagir avec le XRP Ledger.
Le **Testnet** est un reseau de test gratuit ou l'on peut obtenir des XRP fictifs
via un **faucet** (robinet) pour experimenter sans risque.

In [ ]:
# Verification de l'installation de xrpl-py
try:
    import xrpl
    from xrpl.clients import JsonRpcClient
    from xrpl.wallet import generate_faucet_wallet
    from xrpl.models.requests import AccountInfo
    from importlib.metadata import version as pkg_version
    print(f"xrpl-py version : {pkg_version('xrpl-py')}")
    print("Installation OK")
except ImportError:
    print("xrpl-py n'est pas installe.")
    print("Installation : pip install xrpl-py")
    print("Les cellules suivantes ne fonctionneront pas sans cette bibliotheque.")

In [ ]:
# Connexion au Testnet XRP
import xrpl
from xrpl.clients import JsonRpcClient

TESTNET_URL = "https://s.altnet.rippletest.net:51234"

try:
    client = JsonRpcClient(TESTNET_URL)
    # Verifier la connexion en demandant le dernier ledger
    from xrpl.models.requests import Ledger
    response = client.request(Ledger(ledger_index="validated"))
    ledger_index = response.result.get("ledger_index", "inconnu")
    close_time = response.result.get("ledger", {}).get("close_time_human", "inconnu")
    print(f"Connexion au Testnet XRP reussie")
    print(f"  URL           : {TESTNET_URL}")
    print(f"  Ledger valide : #{ledger_index}")
    print(f"  Heure         : {close_time}")
except Exception as e:
    print(f"Impossible de se connecter au Testnet : {e}")
    print("Les cellules suivantes utiliseront des donnees simulees.")
    client = None

In [ ]:
# Creation d'un wallet de test via le faucet
from xrpl.wallet import generate_faucet_wallet

try:
    if client is None:
        raise ConnectionError("Pas de connexion au Testnet")

    print("Demande de XRP au faucet Testnet...")
    print("(cette operation peut prendre 10-20 secondes)\n")

    wallet_1 = generate_faucet_wallet(client, debug=False)
    wallet_2 = generate_faucet_wallet(client, debug=False)

    print("WALLET 1 (Alice)")
    print(f"  Adresse classique : {wallet_1.address}")
    print(f"  Cle publique      : {wallet_1.public_key[:32]}...")
    print(f"  Seed (secret)     : {wallet_1.seed}")
    print()
    print("WALLET 2 (Bob)")
    print(f"  Adresse classique : {wallet_2.address}")
    print(f"  Cle publique      : {wallet_2.public_key[:32]}...")
    print()

    # Consulter le solde
    from xrpl.models.requests import AccountInfo
    acct_info = client.request(AccountInfo(
        account=wallet_1.address,
        ledger_index="validated"
    ))
    balance_drops = int(acct_info.result["account_data"]["Balance"])
    balance_xrp = balance_drops / 1_000_000
    print(f"Solde Alice : {balance_xrp:.6f} XRP ({balance_drops:,} drops)")
    print(f"  (1 XRP = 1,000,000 drops)")

except Exception as e:
    print(f"Erreur faucet : {e}")
    print("\nSimulation avec des donnees fictives :")
    print("  Alice : rHb9CJAWyB4rj91VRWn96DkukG4bwdtyTh (100 XRP)")
    print("  Bob   : rPT1Sjq2YGrBMTttX4GZHjKu9dyfzbpAYe (100 XRP)")
    wallet_1 = None
    wallet_2 = None

### Interpretation

Le faucet Testnet distribue des XRP gratuits pour les tests. Chaque compte XRP
necessite une **reserve de base** (actuellement 10 XRP sur le mainnet) pour exister.
Cette reserve empeche la creation massive de comptes spam.

**Anatomie d'une adresse XRP** :
- Commence par `r` (Base58Check, similaire a Bitcoin)
- 25-34 caracteres
- Derivee de la cle publique Ed25519 ou secp256k1

**Unites** : 1 XRP = 1 000 000 **drops** (l'unite atomique du XRPL).

---

## 3. Transactions XRP

Le XRPL supporte nativement plusieurs types de transactions, sans avoir besoin
de smart contracts :

| Type | Description |
|------|-------------|
| `Payment` | Transfert de XRP ou d'IOU |
| `TrustSet` | Etablir une trust line (autoriser un IOU) |
| `OfferCreate` | Placer un ordre sur le DEX integre |
| `OfferCancel` | Annuler un ordre DEX |
| `EscrowCreate` | Verrouiller des XRP conditionnellement |
| `PaymentChannelCreate` | Ouvrir un canal de micropaiement |

Chaque transaction est signee par la cle privee de l'expediteur et validee
par le consensus en 3-5 secondes.

In [ ]:
# Envoi d'un paiement XRP
import xrpl
from xrpl.models.transactions import Payment
from xrpl.transaction import submit_and_wait
from xrpl.utils import xrp_to_drops

try:
    if wallet_1 is None or wallet_2 is None or client is None:
        raise ConnectionError("Wallets non disponibles")

    # Preparer un paiement de 25 XRP d'Alice vers Bob
    payment_tx = Payment(
        account=wallet_1.address,
        destination=wallet_2.address,
        amount=xrp_to_drops(25),  # 25 XRP en drops
    )

    print("ENVOI DE PAIEMENT XRP")
    print("=" * 60)
    print(f"De      : {wallet_1.address}")
    print(f"Vers    : {wallet_2.address}")
    print(f"Montant : 25 XRP ({xrp_to_drops(25)} drops)")
    print(f"\nSoumission en cours...")

    # Soumettre et attendre la validation
    response = submit_and_wait(payment_tx, client, wallet_1)
    result = response.result

    tx_hash = result.get("hash", "inconnu")
    status = result.get("meta", {}).get("TransactionResult", "inconnu")
    fee_drops = int(result.get("Fee", 0))

    print(f"\nResultat :")
    print(f"  Hash        : {tx_hash}")
    print(f"  Statut      : {status}")
    print(f"  Frais       : {fee_drops} drops ({fee_drops / 1_000_000:.6f} XRP)")
    print(f"  Ledger      : {result.get('ledger_index', 'inconnu')}")

    if status == "tesSUCCESS":
        print("\n-> Transaction validee en quelques secondes !")
    else:
        print(f"\n-> Resultat inattendu : {status}")

except Exception as e:
    print(f"Erreur lors du paiement : {e}")
    print("\nExemple de resultat attendu :")
    print("  Hash   : A1B2C3...  (64 caracteres hex)")
    print("  Statut : tesSUCCESS")
    print("  Frais  : 12 drops (0.000012 XRP)")

In [ ]:
# Creation d'une Trust Line pour un IOU (devise personnalisee)
from xrpl.models.transactions import TrustSet
from xrpl.models.amounts import IssuedCurrencyAmount

try:
    if wallet_1 is None or wallet_2 is None or client is None:
        raise ConnectionError("Wallets non disponibles")

    # Bob autorise Alice a emettre jusqu'a 1000 EUR (IOU)
    trust_set_tx = TrustSet(
        account=wallet_2.address,
        limit_amount=IssuedCurrencyAmount(
            currency="EUR",
            issuer=wallet_1.address,
            value="1000",
        ),
    )

    print("CREATION D'UNE TRUST LINE")
    print("=" * 60)
    print(f"Bob autorise Alice a emettre jusqu'a 1000 EUR")
    print(f"  Compte      : {wallet_2.address} (Bob)")
    print(f"  Emetteur    : {wallet_1.address} (Alice)")
    print(f"  Devise      : EUR")
    print(f"  Limite      : 1000")
    print(f"\nSoumission en cours...")

    response = submit_and_wait(trust_set_tx, client, wallet_2)
    status = response.result.get("meta", {}).get("TransactionResult", "inconnu")

    print(f"  Statut : {status}")

    if status == "tesSUCCESS":
        print("\n-> Trust line creee. Bob peut maintenant recevoir des EUR d'Alice.")
        print("   Les trust lines sont la base du systeme de credit du XRPL.")

except Exception as e:
    print(f"Erreur trust line : {e}")
    print("\nPrincipe d'une trust line :")
    print("  - Un compte CHOISIT de faire confiance a un emetteur")
    print("  - Limite maximale configurable")
    print("  - Permet les paiements en devises (IOU) sur le XRPL")
    print("  - Pas de trust line = impossible de recevoir cet IOU")

In [ ]:
# Creation d'un Offer sur le DEX integre
from xrpl.models.transactions import OfferCreate
from xrpl.models.amounts import IssuedCurrencyAmount
from xrpl.utils import xrp_to_drops

try:
    if wallet_1 is None or client is None:
        raise ConnectionError("Wallets non disponibles")

    # Alice propose de vendre 50 XRP contre 45 EUR
    offer_tx = OfferCreate(
        account=wallet_1.address,
        taker_pays=xrp_to_drops(50),      # Ce que l'acheteur paie : 50 XRP
        taker_gets=IssuedCurrencyAmount(   # Ce que l'acheteur recoit : 45 EUR
            currency="EUR",
            issuer=wallet_1.address,
            value="45",
        ),
    )

    print("CREATION D'UN OFFER (DEX INTEGRE)")
    print("=" * 60)
    print("Le XRPL possede un DEX (Decentralized Exchange) natif.")
    print("Pas besoin de smart contract comme Uniswap sur Ethereum.\n")
    print(f"Alice propose :")
    print(f"  Vend  : 45 EUR (IOU)")
    print(f"  Recoit: 50 XRP")
    print(f"  Taux  : 1 EUR = 1.11 XRP\n")
    print(f"Soumission en cours...")

    response = submit_and_wait(offer_tx, client, wallet_1)
    status = response.result.get("meta", {}).get("TransactionResult", "inconnu")
    print(f"  Statut : {status}")

    if status == "tesSUCCESS":
        print("\n-> Offer placee sur le carnet d'ordres du XRPL")
        print("   Elle sera executee automatiquement si un offer correspondant apparait.")

except Exception as e:
    print(f"Erreur offer : {e}")
    print("\nArchitecture du DEX XRPL :")
    print("  - Carnet d'ordres on-chain (order book)")
    print("  - Matching automatique lors de la soumission")
    print("  - Auto-bridging via XRP (EUR->XRP->USD)")
    print("  - Pas de frais de gas supplementaires")

### Interpretation

Les trois types de transactions ci-dessus illustrent la philosophie du XRPL :
les fonctionnalites financieres sont **integrees au protocole**, pas ajoutees
via des smart contracts.

| Fonctionnalite | Ethereum | XRP Ledger |
|---------------|----------|------------|
| Transfert de tokens | Smart contract ERC-20 | Trust line + Payment natif |
| Echange decentralise | Uniswap (smart contract) | DEX natif (OfferCreate) |
| Depot conditionnel | Smart contract Escrow | EscrowCreate natif |
| Micropaiements | State channels (complexe) | PaymentChannel natif |

**Avantage** : fiabilite, performance, cout minimal.
**Inconvenient** : impossible d'inventer de nouvelles primitives sans modifier le protocole
(avant l'arrivee des Hooks, cf. section 5).

---

## 4. Payment Channels (micropaiements)

Les **Payment Channels** permettent des flux de micropaiements off-chain
entre deux parties, avec un reglement final on-chain.

### Principe

```
1. Alice ouvre un canal : verrouille 100 XRP on-chain
2. Alice envoie des "claims" signes a Bob (off-chain, instantane)
   - Claim 1 : Bob peut reclamer 1 XRP
   - Claim 2 : Bob peut reclamer 2 XRP (remplace claim 1)
   - Claim 3 : Bob peut reclamer 3 XRP (remplace claim 2)
   - ...
3. Bob soumet le dernier claim on-chain pour recevoir ses XRP
4. Le canal est ferme, Alice recupere le reste
```

**Cas d'usage** : streaming de paiements (payer a la seconde pour un service),
microtransactions (IoT, API pay-per-call), et tout scenario ou les frais
de transaction on-chain seraient prohibitifs.

In [ ]:
# Demonstration conceptuelle des Payment Channels
# (La creation reelle necessite un canal ouvert sur le ledger)

import hashlib
import json
import time

print("PAYMENT CHANNEL - SIMULATION CONCEPTUELLE")
print("=" * 60)

# Etape 1 : Ouverture du canal
channel_params = {
    "source": "rAlice...",
    "destination": "rBob...",
    "amount_xrp": 100,
    "settle_delay_seconds": 3600,  # 1 heure pour contester
}

print("1. OUVERTURE DU CANAL")
print(f"   Source      : {channel_params['source']}")
print(f"   Destination : {channel_params['destination']}")
print(f"   Reserve     : {channel_params['amount_xrp']} XRP")
print(f"   Delai       : {channel_params['settle_delay_seconds']}s")
print()

# Etape 2 : Flux de claims off-chain
print("2. FLUX DE CLAIMS OFF-CHAIN")
print("-" * 40)

cumulative_amount = 0
claims = []
for i in range(1, 11):
    cumulative_amount += 0.5  # 0.5 XRP par increment
    claim = {
        "channel_id": "ABC123...",
        "amount_drops": int(cumulative_amount * 1_000_000),
        "signature": hashlib.sha256(
            f"claim:{cumulative_amount}".encode()
        ).hexdigest()[:16] + "...",
    }
    claims.append(claim)
    print(f"   Claim {i:2d} : {cumulative_amount:.1f} XRP"
          f"  (sig: {claim['signature'][:12]}...)")

print()

# Etape 3 : Reglement final
last_claim = claims[-1]
remaining = channel_params["amount_xrp"] - cumulative_amount

print("3. REGLEMENT FINAL ON-CHAIN")
print(f"   Bob soumet le dernier claim : {cumulative_amount:.1f} XRP")
print(f"   Bob recoit                  : {cumulative_amount:.1f} XRP")
print(f"   Alice recupere le reste     : {remaining:.1f} XRP")
print(f"   Total = {cumulative_amount + remaining:.1f} XRP (integre)")
print()
print("-> 10 transferts realises pour le cout d'1 seule transaction on-chain")
print("-> Latence quasi-nulle pour les claims off-chain")

In [ ]:
# Utilisation reelle via xrpl-py (ouverture de canal)
from xrpl.models.transactions import PaymentChannelCreate
from xrpl.utils import xrp_to_drops

try:
    if wallet_1 is None or wallet_2 is None or client is None:
        raise ConnectionError("Wallets non disponibles")

    # Ouverture d'un canal de 20 XRP
    channel_tx = PaymentChannelCreate(
        account=wallet_1.address,
        destination=wallet_2.address,
        amount=xrp_to_drops(20),
        settle_delay=3600,           # 1 heure de delai de reglement
        public_key=wallet_1.public_key,
    )

    print("OUVERTURE D'UN PAYMENT CHANNEL REEL")
    print("=" * 60)
    print(f"  De      : {wallet_1.address}")
    print(f"  Vers    : {wallet_2.address}")
    print(f"  Reserve : 20 XRP")
    print(f"  Delai   : 3600 secondes\n")
    print("Soumission en cours...")

    response = submit_and_wait(channel_tx, client, wallet_1)
    status = response.result.get("meta", {}).get("TransactionResult", "inconnu")
    print(f"  Statut : {status}")

    if status == "tesSUCCESS":
        # Extraire le channel ID des metadonnees
        meta = response.result.get("meta", {})
        affected = meta.get("AffectedNodes", [])
        channel_id = None
        for node in affected:
            created = node.get("CreatedNode", {})
            if created.get("LedgerEntryType") == "PayChannel":
                channel_id = created.get("LedgerIndex")
                break
        if channel_id:
            print(f"  Channel ID : {channel_id}")
        print("\n-> Canal ouvert. Les claims off-chain peuvent commencer.")

except Exception as e:
    print(f"Erreur payment channel : {e}")
    print("\nLe Payment Channel XRPL permet :")
    print("  - Micropaiements instantanes off-chain")
    print("  - Un seul cout de transaction pour ouvrir + fermer")
    print("  - Streaming de paiements (IoT, API, contenu)")

### Interpretation

Les Payment Channels du XRPL sont une solution native aux **micropaiements** :

| Aspect | On-chain classique | Payment Channel |
|--------|-------------------|----------------|
| Cout par transfert | ~0.000012 XRP | **0** (off-chain) |
| Latence | 3-5 secondes | **Instantane** |
| Debit | ~1500 TPS | **Illimite** (off-chain) |
| Securite | Consensus complet | Signature cryptographique |

Bob est protege car il peut soumettre le dernier claim a tout moment.
Alice est protegee par le delai de reglement (`settle_delay`) qui empeche
Bob de reclamer des fonds apres la fermeture.

---

## 5. Hooks : les smart contracts du XRPL

Les **Hooks** sont l'equivalent des smart contracts pour le XRP Ledger.
Introduits via l'amendement `Hooks` (en cours de deploiement), ils permettent
d'attacher une logique programmable aux comptes du XRPL.

### Architecture des Hooks

| Propriete | Ethereum Smart Contract | XRP Hook |
|-----------|------------------------|----------|
| Langage | Solidity / Vyper | **C** (compile en WASM) |
| Execution | EVM | **WebAssembly (WASM)** |
| Declenchement | Appel explicite | **Automatique** (pre/post transaction) |
| Stockage | Storage slots (256 bits) | Hook State (32 bytes key-value) |
| Limites | Gas limit | **Instruction count** |

### Declenchement

Un Hook est declenche **automatiquement** lorsqu'une transaction implique
le compte auquel il est attache :

```
Transaction entrante -> Hook cbak() -> accepter / rejeter
Transaction sortante -> Hook hook() -> modifier / bloquer
```

### Exemple conceptuel de Hook (C)

```c
#include "hookapi.h"

// Hook qui limite les paiements sortants a 100 XRP maximum
int64_t hook(uint32_t reserved) {
    // Lire le montant de la transaction
    int64_t amount = otxn_amount();
    
    // Limite de 100 XRP (en drops)
    int64_t limit = 100000000; // 100 * 1,000,000 drops
    
    if (amount > limit) {
        // Rejeter la transaction
        rollback(SBUF("Payment exceeds 100 XRP limit"), 1);
    }
    
    // Accepter la transaction
    accept(SBUF("Payment approved"), 0);
    return 0;
}
```

## API Hook principale

| Fonction | Description |
|----------|-------------|
| `hook()` | Point d'entree pour les transactions sortantes |
| `cbak()` | Callback pour les transactions entrantes |
| `accept()` | Approuver la transaction |
| `rollback()` | Rejeter la transaction |
| `otxn_amount()` | Montant de la transaction courante |
| `otxn_field()` | Lire un champ de la transaction |
| `state_set()` | Ecrire dans le stockage du Hook |
| `state()` | Lire le stockage du Hook |
| `emit()` | Emettre une transaction depuis le Hook |

> **Note** : Les Hooks sont en cours de deploiement sur le mainnet XRPL.
> Ils sont disponibles sur le testnet Hooks (hooks-testnet-v3.xrpl-labs.com).
> La compilation C -> WASM necessite un toolchain specifique (wasmcc).

In [ ]:
# Les Hooks ne sont pas encore deployables via xrpl-py standard.
# Voici une representation structurelle d'un Hook.

print("ANATOMIE D'UN HOOK XRPL")
print("=" * 60)

hook_example = {
    "nom": "Limite de paiement sortant",
    "description": "Rejette tout paiement sortant > 100 XRP",
    "langage": "C -> WebAssembly (WASM)",
    "declenchement": "Automatique sur chaque transaction sortante",
    "fonctions": {
        "hook()": "Verifie le montant, accept/rollback",
        "cbak()": "Non utilise (transactions entrantes)",
    },
    "etat_stocke": {
        "limite": "100 XRP (modifiable par le proprietaire)",
        "compteur_rejets": "Nombre de transactions rejetees",
    },
    "taille_wasm": "~2 Ko",
    "cout_installation": "~2 XRP (reserve d'objet)",
}

for key, val in hook_example.items():
    if isinstance(val, dict):
        print(f"\n{key} :")
        for k, v in val.items():
            print(f"  {k:25s} : {v}")
    else:
        print(f"{key:25s} : {val}")

print()
print("Cycle de vie d'un Hook :")
print("  1. Ecriture en C avec hookapi.h")
print("  2. Compilation en WASM (wasmcc)")
print("  3. Installation via SetHook transaction")
print("  4. Declenchement automatique a chaque transaction")
print("  5. Desinstallation via SetHook (wasm vide)")

### Comparaison : Hooks vs Smart Contracts Ethereum

| Aspect | Smart Contract Ethereum | Hook XRPL |
|--------|------------------------|-----------|  
| Deploiement | Adresse propre | Attache a un compte existant |
| Appel | Explicite (tx vers le contrat) | Implicite (toute tx du compte) |
| Turing-complet | Oui | **Non** (pas de boucles infinies) |
| Composabilite | Appels inter-contrats | Limitee (emit) |
| Maturite | Production depuis 2015 | En cours de deploiement |

Les Hooks ne remplacent pas les smart contracts Ethereum pour les cas d'usage
complexes (DeFi avancee, NFT), mais ils couvrent efficacement les scenarios
lies aux **paiements** : limites, compliance, routage automatique.

---

## 6. Exercice : scenario de paiement cross-currency

Implementez un scenario complet de paiement cross-currency sur le XRPL Testnet :

1. Creer 3 wallets (Alice, Bob, Gateway)
2. Gateway etablit des trust lines pour EUR et USD
3. Alice et Bob etablissent des trust lines vers Gateway
4. Gateway emet des EUR a Alice et des USD a Bob
5. Alice paie Bob en EUR, qui arrive en USD (via le DEX)

Ce scenario illustre le cas d'usage principal du XRPL : le **paiement
transfrontalier** avec conversion automatique de devises.

In [ ]:
# Exercice : Paiement cross-currency via trust lines et DEX

import xrpl
from xrpl.clients import JsonRpcClient
from xrpl.wallet import generate_faucet_wallet
from xrpl.models.transactions import TrustSet, Payment, OfferCreate
from xrpl.models.amounts import IssuedCurrencyAmount
from xrpl.transaction import submit_and_wait
from xrpl.utils import xrp_to_drops


def cross_currency_scenario(client):
    """Scenario complet de paiement cross-currency.

    Etapes :
    1. Creer 3 wallets via le faucet (Alice, Bob, Gateway)
    2. Gateway cree des trust lines EUR et USD
    3. Alice etablit une trust line EUR vers Gateway
    4. Bob etablit une trust line USD vers Gateway
    5. Gateway emet 500 EUR a Alice (Payment IOU)
    6. Gateway emet 500 USD a Bob (Payment IOU)
    7. Creer un offer EUR/USD sur le DEX pour etablir un taux
    8. Alice envoie 100 EUR a Bob, qui arrive en ~110 USD
       (utiliser SendMax et DeliverMin pour le cross-currency)
    9. Afficher les soldes finaux de tous les comptes

    Returns:
        dict: Soldes finaux de chaque participant
    """
    # TODO : Implementer le scenario complet
    raise NotImplementedError(
        "Implementez le scenario de paiement cross-currency.\n"
        "Indice : utilisez IssuedCurrencyAmount pour les montants IOU,\n"
        "et le champ send_max du Payment pour le cross-currency."
    )


# Test (decommentez apres implementation)
# try:
#     if client is not None:
#         result = cross_currency_scenario(client)
#         print("Scenario cross-currency termine avec succes")
#         for account, balances in result.items():
#             print(f"  {account}: {balances}")
#     else:
#         print("Client non disponible - connectez-vous au Testnet")
# except NotImplementedError as e:
#     print(f"Exercice non implemente : {e}")
# except Exception as e:
#     print(f"Erreur : {e}")

---

## 7. Resume

| Fonctionnalite | Mecanisme XRPL | Equivalent Ethereum |
|---------------|----------------|--------------------|
| **Consensus** | UNL (3-5 sec, deterministe) | PoS Casper (~12 sec) |
| **Tokens** | Trust Lines natives | Smart contract ERC-20 |
| **DEX** | Order book integre au protocole | Uniswap/smart contract |
| **Micropaiements** | Payment Channels natifs | State channels (complexe) |
| **Smart Contracts** | Hooks (C/WASM) | Solidity/EVM |
| **Escrow** | Natif (time-lock, conditions) | Smart contract Escrow |
| **Frais** | ~0.000012 XRP (~$0.00001) | Variable (gas, congestion) |

### Points cles

1. Le XRPL est optimise pour les **paiements** : rapide, peu couteux, deterministe
2. Les fonctionnalites financieres sont **natives au protocole**, pas dans des smart contracts
3. Les **trust lines** permettent la creation de tokens (IOU) sans deployer de code
4. Le **DEX integre** offre un echange decentralise sans intermediaire logiciel
5. Les **Hooks** (C/WASM) ajoutent la programmabilite manquante au XRPL
6. Le consensus UNL sacrifie un peu de decentralisation pour la **performance**

### Ressources

- [Documentation XRPL](https://xrpl.org/docs.html)
- [xrpl-py GitHub](https://github.com/XRPLF/xrpl-py)
- [XRPL Testnet Faucet](https://xrpl.org/xrp-testnet-faucet.html)
- [Hooks Documentation](https://xrpl-hooks.readme.io/)

---

**Notebook suivant** : [SC-20-Bitcoin-Scripting](SC-20-Bitcoin-Scripting.ipynb) - Le langage Script de Bitcoin

---

[<< Vyper](SC-18-Vyper.ipynb) | [Bitcoin Scripting >>](SC-20-Bitcoin-Scripting.ipynb)